# Testing the SIarea ice mask

Quick sanity check for the `--ice-mask` flag added to `zarr_to_netcdf.py`.

This notebook:
1. Loads one or more variables from their zarr stores on S3 (unmasked source of truth).
2. Loads **SIarea** from `icearea.zarr` on S3.
3. Applies the mask (`SIarea > 0 → NaN`) in-notebook.
4. Plots before / SIarea / after for each variable (global + polar close-ups).
5. Optionally compares against an exported NetCDF.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cmocean.cm as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import yaml

import dbof.dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.dataset_creation.zarr_grid_global as zarr_grid
import dbof.io.filesystems as filesystems

## Configuration

`VARIABLES` is a list of dicts, one per variable to test. Each entry
specifies the channel name, zarr store, colormap, label, and norm type.
Add or remove entries to test different fields.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# USER SETTINGS
# ════════════════════════════════════════════════════════════════════

# Variables to test.  Each dict needs:
#   channel    : channel name in the zarr store
#   zarr_store : dataset_name of the zarr store
#   cmap       : cmocean colormap name
#   label      : colorbar label
#   norm       : 'log' for LogNorm, 'linear' for Normalize,
#                'diverging' for TwoSlopeNorm centred at 0
#   netcdf     : (optional) path to an exported NetCDF for comparison
VARIABLES = [
    {
        "channel":    "N2_sfc",
        "zarr_store": "stratification.zarr",
        "cmap":       "tempo",
        "label":      "N\u00b2 (s\u207b\u00b2)",
        "norm":       "log",
        "netcdf":     (
            "/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/vtest/"
            "20121109_120000/LLC4320_2012-11-09T12_00_00_N2_sfc.nc"
        ),
    },
    {
        "channel":    "Theta_sfc",
        "zarr_store": "native_fields.zarr",
        "cmap":       "thermal",
        "label":      "Potential temperature (\u00b0C)",
        "norm":       "linear",
        "netcdf":     None,
    },
]

ICE_ZARR_STORE = "icearea.zarr"

# S3 coordinates (must match the export run)
S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "properties"
RUN_ID      = "testing_019"
DATE_PREFIX = "20121109_120000"

# Downsample factor for plotting
DS = 20

## Load SIarea and grid

In [ ]:
fs, fs_synch = filesystems.create_s3_filesystems(S3_ENDPOINT)

# ── SIarea ─────────────────────────────────────────────────────────────
ice_reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=ICE_ZARR_STORE, date_prefix=DATE_PREFIX, fs=fs,
)
siarea = ice_reader.get_channel_snapshot(0, "SIarea").astype(np.float32)
ice_mask = siarea > 0

print(f"SIarea loaded from {ICE_ZARR_STORE}")
print(f"  shape: {siarea.shape}")
print(f"  range: [{np.nanmin(siarea):.4g}, {np.nanmax(siarea):.4g}]")
print(f"  ice-covered points (SIarea > 0): {ice_mask.sum():,} / {ice_mask.size:,} "
      f"({ice_mask.mean():.2%})")

# ── Grid ───────────────────────────────────────────────────────────────
path_to_config = "../../configs/data_access/global_depth.yaml"
with open(path_to_config) as f:
    cfg = yaml.safe_load(f)
grid_cfg = cfg["grid_access"]

fs_grid, _ = filesystems.create_s3_filesystems(grid_cfg["s3_endpoint"])
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=grid_cfg["bucket"], folder=grid_cfg["folder"],
    dataset_name=grid_cfg["dataset_name"], fs=fs_grid,
)
XC = grid_reader.lon
YC = grid_reader.lat
XC_g = XC[::DS, ::DS]
YC_g = YC[::DS, ::DS]
print(f"Grid loaded: XC {XC.shape}")

# ── Downsampled SIarea for reuse across plots ──────────────────────
siarea_ds = siarea[::DS, ::DS]

## Load all variables from zarr

In [ ]:
# Cache opened readers so we don't re-open the same store twice.
_reader_cache = {}

for v in VARIABLES:
    store = v["zarr_store"]
    ch    = v["channel"]

    if store not in _reader_cache:
        _reader_cache[store] = zarr_dataset.GlobalZarrDatasetReader(
            bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
            dataset_name=store, date_prefix=DATE_PREFIX, fs=fs,
        )
    reader = _reader_cache[store]

    arr = reader.get_channel_snapshot(0, ch).astype(np.float32)
    v["arr_orig"]   = arr
    v["arr_masked"] = np.where(ice_mask, np.nan, arr)

    n_new_nan = (np.isnan(v["arr_masked"]) & ~np.isnan(arr)).sum()
    print(f"{ch} from {store}:")
    print(f"  finite range: [{np.nanmin(arr):.4g}, {np.nanmax(arr):.4g}]")
    print(f"  NaN: {np.isnan(arr).mean():.2%} -> {np.isnan(v['arr_masked']).mean():.2%}  "
          f"(+{n_new_nan:,} from ice mask)")

## Helper: build norm and colormap for a variable

In [ ]:
def make_norm_and_cmap(v, data_ds):
    """Return (norm, cmap) for a variable config dict."""
    cmap = getattr(cmo, v["cmap"])
    finite = data_ds[np.isfinite(data_ds)]

    if v["norm"] == "log":
        pos = finite[finite > 0]
        lo, hi = (np.nanpercentile(pos, [1, 99]) if pos.size > 0
                  else (1e-8, 1e-3))
        norm = mcolors.LogNorm(vmin=lo, vmax=hi)
    elif v["norm"] == "diverging":
        lo, hi = np.nanpercentile(finite, [1, 99])
        vlim = max(abs(lo), abs(hi))
        norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vlim, vmax=vlim)
    else:  # linear
        lo, hi = np.nanpercentile(finite, [1, 99])
        norm = mcolors.Normalize(vmin=lo, vmax=hi)

    return norm, cmap

## Global maps: Original vs. SIarea vs. Masked

One row of three panels per variable.

In [ ]:
for v in VARIABLES:
    ch = v["channel"]
    orig_ds   = v["arr_orig"][::DS, ::DS]
    masked_ds = v["arr_masked"][::DS, ::DS]
    norm, cmap = make_norm_and_cmap(v, orig_ds)

    fig, axes = plt.subplots(
        1, 3, figsize=(24, 7),
        subplot_kw={"projection": ccrs.Robinson()},
    )

    # Panel 1: Original
    ax = axes[0]
    im = ax.pcolormesh(XC_g, YC_g, orig_ds, cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(), shading="nearest")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color="k")
    ax.gridlines(linewidth=0.3, color="gray", alpha=0.5)
    ax.set_title(f"{ch} \u2014 original (from zarr)", fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, shrink=0.7,
                 label=v["label"], orientation="horizontal")

    # Panel 2: SIarea
    ax = axes[1]
    im = ax.pcolormesh(XC_g, YC_g, siarea_ds, cmap=cmo.ice, vmin=0, vmax=1,
                       transform=ccrs.PlateCarree(), shading="nearest")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color="k")
    ax.gridlines(linewidth=0.3, color="gray", alpha=0.5)
    ax.set_title("SIarea (0 = open water, >0 = ice)", fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, shrink=0.7,
                 label="Sea-ice area fraction", orientation="horizontal")

    # Panel 3: Masked
    ax = axes[2]
    im = ax.pcolormesh(XC_g, YC_g, masked_ds, cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(), shading="nearest")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color="k")
    ax.gridlines(linewidth=0.3, color="gray", alpha=0.5)
    ax.set_title(f"{ch} \u2014 ice-masked (SIarea > 0 \u2192 NaN)", fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, shrink=0.7,
                 label=v["label"], orientation="horizontal")

    plt.suptitle(
        f"{ch}  |  run_id={RUN_ID}  |  {DATE_PREFIX}  |  {DS}\u00d7 downsampled",
        fontsize=14, y=1.02,
    )
    plt.tight_layout()
    plt.show()

## Polar close-ups (Arctic & Antarctic)

One 2\u00d73 figure per variable (rows = Arctic / Antarctic,
cols = original / SIarea / masked).

In [ ]:
for v in VARIABLES:
    ch = v["channel"]
    orig_ds   = v["arr_orig"][::DS, ::DS]
    masked_ds = v["arr_masked"][::DS, ::DS]
    norm, cmap = make_norm_and_cmap(v, orig_ds)

    fig, axes = plt.subplots(
        2, 3, figsize=(21, 14),
        subplot_kw={"projection": ccrs.Orthographic(0, 90)},
    )

    for row, (lat0, label) in enumerate([(90, "Arctic"), (-90, "Antarctic")]):
        for col, ax in enumerate(axes[row]):
            ax.set_global()
            proj = ccrs.Orthographic(0, lat0)
            pos = ax.get_position()
            ax.remove()
            ax = fig.add_axes(pos, projection=proj)
            axes[row, col] = ax

            if col == 0:
                data, title = orig_ds, f"{label}: original"
                im = ax.pcolormesh(XC_g, YC_g, data, cmap=cmap, norm=norm,
                                   transform=ccrs.PlateCarree(), shading="nearest")
            elif col == 1:
                data, title = siarea_ds, f"{label}: SIarea"
                im = ax.pcolormesh(XC_g, YC_g, data, cmap=cmo.ice, vmin=0, vmax=1,
                                   transform=ccrs.PlateCarree(), shading="nearest")
            else:
                data, title = masked_ds, f"{label}: ice-masked"
                im = ax.pcolormesh(XC_g, YC_g, data, cmap=cmap, norm=norm,
                                   transform=ccrs.PlateCarree(), shading="nearest")

            ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color="k")
            ax.gridlines(linewidth=0.3, color="gray", alpha=0.5)
            ax.set_title(title, fontsize=11)

    plt.suptitle(
        f"Polar close-ups \u2014 {ch}  |  {RUN_ID}  |  {DATE_PREFIX}",
        fontsize=14, y=1.02,
    )
    plt.tight_layout()
    plt.show()

## Difference maps: what the mask removed

For each variable, show only the points that went from finite to NaN.

In [ ]:
for v in VARIABLES:
    ch = v["channel"]
    newly_masked = np.isnan(v["arr_masked"]) & ~np.isnan(v["arr_orig"])
    diff_arr = np.where(newly_masked, v["arr_orig"], np.nan)[::DS, ::DS]

    orig_ds = v["arr_orig"][::DS, ::DS]
    norm, _ = make_norm_and_cmap(v, orig_ds)

    fig = plt.figure(figsize=(16, 8))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

    im = ax.pcolormesh(XC_g, YC_g, diff_arr, cmap=cmo.matter, norm=norm,
                       transform=ccrs.PlateCarree(), shading="nearest")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color="k")
    ax.add_feature(cfeature.LAND, facecolor="lightgray", edgecolor="none")
    ax.gridlines(linewidth=0.3, color="gray", alpha=0.5)
    ax.set_title(
        f"{ch} \u2014 points removed by ice mask  |  {newly_masked.sum():,} points",
        fontsize=13,
    )
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, shrink=0.7,
                 label=f"{ch} original value at masked points",
                 orientation="horizontal")
    plt.tight_layout()
    plt.show()

## Compare against exported NetCDF(s)

For each variable with a `netcdf` path, load the file and check whether
the ice mask was applied during export.

In [ ]:
for v in VARIABLES:
    ch = v["channel"]
    nc_path = v.get("netcdf")
    if nc_path is None:
        print(f"{ch}: no NetCDF path configured \u2014 skipping.\n")
        continue

    ds_nc = xr.open_dataset(nc_path)
    arr_nc = ds_nc[ch].values

    nc_nan   = np.isnan(arr_nc)
    zarr_nan = np.isnan(v["arr_orig"])
    extra    = nc_nan & ~zarr_nan

    print(f"{ch}: {nc_path}")
    print(f"  NaN fraction (zarr):   {zarr_nan.mean():.2%}")
    print(f"  NaN fraction (NetCDF): {nc_nan.mean():.2%}")
    print(f"  Extra NaN in NetCDF:   {extra.sum():,}")

    if extra.sum() > 0:
        overlap = extra & ice_mask
        print(f"  Of those, {overlap.sum():,} overlap with SIarea > 0")
        print(f"  Ice mask points: {ice_mask.sum():,}")
        if overlap.sum() == extra.sum() == ice_mask.sum():
            print("  --> Ice mask applied correctly.")
        elif overlap.sum() == extra.sum():
            print("  --> All extra NaN from ice (some ice points were already NaN/land).")
        else:
            print("  --> MISMATCH: some extra NaN are NOT from the ice mask.")
    else:
        print("  --> No extra NaN \u2014 ice mask was NOT applied during export.")
    print()

## Statistics: before vs. after (all variables)

In [ ]:
for v in VARIABLES:
    ch = v["channel"]
    arr_o = v["arr_orig"]
    arr_m = v["arr_masked"]
    o_fin = arr_o[np.isfinite(arr_o)]
    m_fin = arr_m[np.isfinite(arr_m)]

    print(f"=== {ch} ({v['zarr_store']}) ===")
    print(f"{'Metric':<30s}  {'Original':>14s}  {'Ice-masked':>14s}")
    print("\u2500" * 62)
    print(f"{'Total points':<30s}  {arr_o.size:>14,}  {arr_m.size:>14,}")
    print(f"{'Finite points':<30s}  {o_fin.size:>14,}  {m_fin.size:>14,}")
    print(f"{'NaN points':<30s}  {np.isnan(arr_o).sum():>14,}  {np.isnan(arr_m).sum():>14,}")
    print(f"{'NaN fraction':<30s}  {np.isnan(arr_o).mean():>14.2%}  {np.isnan(arr_m).mean():>14.2%}")
    print(f"{'New NaN from ice':<30s}  {'\u2014':>14s}  {(np.isnan(arr_m) & ~np.isnan(arr_o)).sum():>14,}")
    if o_fin.size > 0 and m_fin.size > 0:
        print(f"{'Min':<30s}  {o_fin.min():>14.4g}  {m_fin.min():>14.4g}")
        print(f"{'Max':<30s}  {o_fin.max():>14.4g}  {m_fin.max():>14.4g}")
        print(f"{'Mean':<30s}  {o_fin.mean():>14.4g}  {m_fin.mean():>14.4g}")
        print(f"{'Std':<30s}  {o_fin.std():>14.4g}  {m_fin.std():>14.4g}")
    print()